In [46]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [47]:
spx = pd.read_csv('../Data/stock_data/^SP500TR.csv')
rf = pd.read_csv('../Data/risk_free_rates/risk_free_USA.csv', skiprows = 9)
cg = pd.read_csv('../Structure_models/CG_BA.csv')
df = pd.read_csv('../Data/aggregation/BA_for_CG.csv')

In [48]:
def monthly(x):
    return x.assign(Date=pd.to_datetime(x.Date)).groupby(pd.Grouper(key="Date", freq="ME"))

def rate(T):
    a, b = T / rf.TAU1, T / rf.TAU2
    f, g = -np.expm1(-a) / a, -np.expm1(-b) / b
    return (rf.BETA0 + rf.BETA1*f + rf.BETA2*(f-np.exp(-a)) + rf.BETA3*(g-np.exp(-b))) / 100

y6m = rate(0.5)
curve = monthly(rf.assign(y6m=y6m, PREM=rate(10)-y6m))[["y6m", "PREM"]].last()
rs = monthly(cg.assign(RS=cg.Z_spread-cg.CDS_IV)).RS.last()
volume = monthly(df).Volume.sum()
R = monthly(spx).Close.last().pct_change(fill_method=None).rename("R")
VRP = monthly(df.assign(VRP=df.IV-df.sigma_60)).VRP.last()
R_BA = monthly(df)["Adj Close"].last().pct_change(fill_method=None).rename("R_BA")

d = pd.concat([
    rs.diff().rename("dRS"), np.log(volume).diff().rename("dlnVolume"),
    curve.y6m, curve.PREM.diff().rename("dPREM"), R, R.pow(2).diff().rename("dR2"), VRP, R_BA
], axis=1, join="inner").dropna().reset_index()

In [49]:
d

,Date,dRS,dlnVolume,y6m,dPREM,R,dR2,VRP,R_BA
0,2018-02-28,-7.098026,-0.081423,0.019201,0.000150,-0.036856,-0.001920,-0.023902,0.027169
1,2018-03-31,10.811623,0.177078,0.019965,-0.002324,-0.025414,-0.000712,-0.053477,-0.094779
2,2018-04-30,3.219498,-0.289374,0.020924,0.001020,0.003837,-0.000631,-0.088143,0.017324
3,2018-05-31,3.556136,-0.191124,0.021336,-0.001505,0.024082,0.000565,-0.046042,0.061029
4,2018-06-30,7.859065,-0.015535,0.020973,0.000709,0.006154,-0.000542,-0.008873,-0.047279
...,...,...,...,...,...,...,...,...,...
78,2024-08-31,5.520070,-0.007336,0.047510,0.001225,0.024256,0.000440,0.019978,-0.088458
79,2024-09-30,-47.662512,0.400261,0.043519,0.002878,0.021357,-0.000132,0.009278,-0.124899
80,2024-10-31,-31.808702,0.569305,0.044340,0.004138,-0.009068,-0.000374,0.043130,-0.017956
81,2024-11-30,4.352227,-0.229187,0.044288,-0.001005,0.058701,0.003364,0.058556,0.041056


In [50]:
X = sm.add_constant(d[["dlnVolume", "y6m", "dPREM", "R", "dR2"]])
y = d["dRS"]
model = sm.OLS(y, X).fit(cov_type="HC1")
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    dRS   R-squared:                       0.198
Model:                            OLS   Adj. R-squared:                  0.146
Method:                 Least Squares   F-statistic:                     3.261
Date:                Fri, 20 Nov 2026   Prob (F-statistic):             0.0101
Time:                        23:54:07   Log-Likelihood:                -382.02
No. Observations:                  83   AIC:                             776.0
Df Residuals:                      77   BIC:                             790.6
Df Model:                           5                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.1437      5.509     -0.752      0.4

In [51]:
X = sm.add_constant(d[["dlnVolume", "y6m", "dPREM", "R", "dR2", "VRP"]])
y = d["dRS"]
model = sm.OLS(y, X).fit(cov_type="HC1")
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    dRS   R-squared:                       0.272
Model:                            OLS   Adj. R-squared:                  0.214
Method:                 Least Squares   F-statistic:                     5.140
Date:                Fri, 20 Nov 2026   Prob (F-statistic):           0.000176
Time:                        23:54:07   Log-Likelihood:                -378.03
No. Observations:                  83   AIC:                             770.1
Df Residuals:                      76   BIC:                             787.0
Df Model:                           6                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.4696      5.798      0.253      0.8

In [52]:
X = sm.add_constant(d[["dlnVolume", "y6m", "dPREM", "R", "dR2", "VRP", "R_BA"]])
y = d["dRS"]
model = sm.OLS(y, X).fit(cov_type="HC1")
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    dRS   R-squared:                       0.307
Model:                            OLS   Adj. R-squared:                  0.242
Method:                 Least Squares   F-statistic:                     4.371
Date:                Fri, 20 Nov 2026   Prob (F-statistic):           0.000413
Time:                        23:54:07   Log-Likelihood:                -375.97
No. Observations:                  83   AIC:                             767.9
Df Residuals:                      75   BIC:                             787.3
Df Model:                           7                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.0957      5.566      0.377      0.7

In [53]:
spreads = monthly(cg.dropna(subset=["Z_spread", "CDS_IV"]))[["Z_spread", "CDS_IV"]].last().diff().loc[d.Date].reset_index(drop=True)
model_z = sm.OLS(spreads.Z_spread, X).fit(cov_type="HC1")
print(model_z.summary())

                            OLS Regression Results                            
Dep. Variable:               Z_spread   R-squared:                       0.443
Model:                            OLS   Adj. R-squared:                  0.391
Method:                 Least Squares   F-statistic:                     1.634
Date:                Fri, 20 Nov 2026   Prob (F-statistic):              0.139
Time:                        23:54:08   Log-Likelihood:                -409.84
No. Observations:                  83   AIC:                             835.7
Df Residuals:                      75   BIC:                             855.0
Df Model:                           7                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.1419      4.565     -0.250      0.8

In [54]:
model_cds = sm.OLS(spreads.CDS_IV, X).fit(cov_type="HC1")
print(model_cds.summary())

                            OLS Regression Results                            
Dep. Variable:                 CDS_IV   R-squared:                       0.651
Model:                            OLS   Adj. R-squared:                  0.618
Method:                 Least Squares   F-statistic:                     7.232
Date:                Fri, 20 Nov 2026   Prob (F-statistic):           1.31e-06
Time:                        23:54:09   Log-Likelihood:                -399.34
No. Observations:                  83   AIC:                             814.7
Df Residuals:                      75   BIC:                             834.0
Df Model:                           7                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.2376      5.850     -0.553      0.5